# 03 - Output Parser


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Understand raw text vs structured output.
- Parse JSON and Pydantic objects.
- Know when parsers are useful and when native structured output is better.
- Validate model outputs before downstream use.

## Core AI concept

LLM text is flexible; applications usually need predictable data. An output parser converts model output into a Python structure. Modern models often support native structured output, which can be preferable to asking the model to print JSON and then repairing it.

## Architecture / mental model

```text
LLM response
    ↓
Parser / native structured output
    ↓
Validated object
    ↓
Application logic
```

## Connection to the supplied learning material

The supplied material focuses on retrieval and tool schemas. This notebook fills the important bridge between generated text and reliable programmatic data.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

In [ ]:
%pip install -qU pydantic langchain-core

### JSON-style parsing

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
raw = '{"name": "RAG", "score": 0.95}'
print(parser.parse(raw))

### Pydantic parsing

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Evaluation(BaseModel):
    score: float = Field(ge=0, le=1)
    comment: str

pydantic_parser = PydanticOutputParser(pydantic_object=Evaluation)
raw = '{"score": 0.9, "comment": "The answer is grounded."}'
result = pydantic_parser.parse(raw)

print(result)
print(type(result))

### Native structured output (preferred when supported)

In [ ]:
# Conceptual example with a model that supports structured output.
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
structured_llm = llm.with_structured_output(Evaluation)

# result = structured_llm.invoke(
#     "Score this answer from 0 to 1 and explain briefly."
# )
# print(result)

## Evaluation

Test whether output always validates against the schema. Track parse failures, missing fields, wrong types, and unexpected values.

## Improvement ideas

Prefer native structured output when supported, then add application-side validation. Never trust model-generated JSON just because it looks pretty.

## Practice exercises

1. Add an enum field.
2. Validate a list of objects.
3. Create a retry strategy for invalid output.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create

- LangChain output parsers: https://reference.langchain.com/python/langchain-core/output_parsers